# Stage A — training on a free Colab GPU

Runs Stage 6 (training) and Stage 10 (evaluation) for the SIH 26143 oil-spill detector.

**Before you start:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

At the end you download `best.pt`, drop it in `stage_a/checkpoints/` locally, and run
`python -m stage_a.cli detect ...` to produce the per-case detection artifacts the
backend serves when `STAGE_A_MODE=real`.

Nothing in this notebook reports a metric it did not measure. The numbers printed by the
evaluation cell are the only detection numbers that may appear in the final report.

## 1. Confirm the GPU

If this prints `cpu`, stop and switch the runtime — training on Colab's CPU will not finish.

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('\nNo GPU. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')

## 2. Get the Stage A code

Point `REPO_URL` at the project repo. If it is private, either use a personal access
token in the URL or upload the `stage_a/` folder to the Colab file browser and skip this
cell.

In [ ]:
REPO_URL = 'https://github.com/Garvit1512/sih.git'
BRANCH = 'feature/stage-a-detection'

import os

if not os.path.exists('/content/sih'):
    !git clone --branch {BRANCH} {REPO_URL} /content/sih

%cd /content/sih
!ls stage_a

In [ ]:
!pip install -q segmentation-models-pytorch rasterio
print('done')

## 3. Get the corpus

The public 5-class SAR oil-spill dataset (Krestenitis et al.).

**Kaggle route:** upload your `kaggle.json` API token when prompted, then run the
download cell. Get the token from kaggle.com → Account → Create New API Token.

**Manual route:** if you already have the archive, upload it to `/content/` and unzip it
to `stage_a/data/raw/oil-spill` instead — the loader only needs `train/` and `test/`
directories with `images/` and `labels*/` inside them.

In [ ]:
import os

from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(next(iter(uploaded.values())))
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('kaggle credentials in place')

In [ ]:
DATA_ROOT = '/content/sih/stage_a/data/raw/oil-spill'

!pip install -q kaggle
!mkdir -p {DATA_ROOT}

# Replace with the dataset slug you are using. Verify the archive's licence and terms
# before use, and record which mirror you took it from — Stage 3's provenance matters.
DATASET_SLUG = 'nabilsherif/oil-spill'

!kaggle datasets download -d {DATASET_SLUG} -p /content --unzip || echo 'Download failed — check the slug, your token, and the dataset licence.'
!ls /content

In [ ]:
# Point DATA_ROOT at whichever directory actually contains train/ and test/.
# This cell fails loudly rather than training on an empty dataset.
import sys

sys.path.insert(0, '/content/sih')

from pathlib import Path

from stage_a.dataset import resolve_split

train_split = resolve_split(Path(DATA_ROOT), 'train')
test_split = resolve_split(Path(DATA_ROOT), 'test')

print(f'train: {len(train_split.pairs)} image/label pairs from {train_split.images_dir}')
print(f'test:  {len(test_split.pairs)} image/label pairs from {test_split.images_dir}')

## 4. Check the label palette before training

If this reports unmapped colours, **stop and fix the palette in `stage_a/config.py`**.
Unmapped pixels default to `sea_surface`, which inflates the majority class and quietly
corrupts every metric produced downstream — with no other symptom.

In [ ]:
import numpy as np
from PIL import Image

from stage_a.preprocess import class_distribution, rgb_to_class_indices

sample_masks = []
for _, label_path in train_split.pairs[:25]:
    array = np.array(Image.open(label_path))
    if array.ndim == 2:
        sample_masks.append(array)
        continue
    indices, report = rgb_to_class_indices(array)
    sample_masks.append(indices)
    if not report.is_clean:
        print(f'{label_path.name}: {report.unmapped_pixels} unmapped px, colours {report.unmapped_colours[:3]}')

print('\nclass distribution over the sample:')
for name, share in class_distribution(sample_masks).items():
    print(f'  {name:<14}{share*100:6.2f}%')

## 5. Train (Stage 6)

Checkpoints are selected on **oil-spill IoU**, not loss and not mean IoU — the mean is
dominated by sea surface, so selecting on it produces a model that is best at the easy
part.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-7s %(message)s', datefmt='%H:%M:%S')

from stage_a.config import TrainingConfig
from stage_a.train import train

config = TrainingConfig()
config.epochs = 40
config.batch_size = 16   # T4 handles 16 at 256px; drop to 8 if you hit OOM
config.device = 'cuda'

summary = train(Path(DATA_ROOT), config=config)

print('\nbest oil-spill IoU:', summary['best_oil_spill_iou'])
if summary.get('validation_caveat'):
    print('CAVEAT:', summary['validation_caveat'])

## 6. Evaluate on the official held-out split (Stage 10)

These are the detection numbers for the evaluation section. Per-class **and** mean, with
the class distribution beside them so the mean can be read correctly.

Copy this output verbatim. Do not round it up, and do not quote the mean without the
per-class breakdown.

In [ ]:
from torch.utils.data import DataLoader

from stage_a.dataset import SARSegmentationDataset
from stage_a.evaluate import evaluate_model
from stage_a.model import load_checkpoint

model, payload = load_checkpoint(Path('stage_a/checkpoints/best.pt'), device='cuda')
dataset = SARSegmentationDataset(Path(DATA_ROOT), 'test', tile_size=payload['tile_size'], augment=False)
loader = DataLoader(dataset, batch_size=16, shuffle=False)

report = evaluate_model(model, loader, device='cuda')
print(report.render())

report.save(Path('stage_a/artifacts/detection_metrics.json'))

## 7. Download the checkpoint and metrics

Put `best.pt` in `stage_a/checkpoints/` locally. Commit `detection_metrics.json` — it is
small, and it is the evidence behind whatever number ends up on the evaluation slide.

In [ ]:
from google.colab import files

files.download('stage_a/checkpoints/best.pt')
files.download('stage_a/artifacts/detection_metrics.json')
files.download('stage_a/checkpoints/training_summary.json')